# Predicting Bitcoin Prices from Chainlet Data, Explained with SHAP

**CAP 5619, AI for Finance. Alex Hubenthal.**

This notebook predicts the daily Bitcoin price for every day of December 2017, one day ahead, from
lagged prices and chainlet features, and explains the model with SHAP. The data are the three files
provided with the course: the occurrence chainlets, the amount chainlets and the daily price.

Constraints the notebook is built around:

* Chainlet features come from both the occurrence file and the amount file.
* RMSE is computed in USD over the 31 days of December 2017.
* Dates are parsed with datetime functions and never sliced as strings. `year`, `day` and `date` are
  never model inputs.
* The horizon is one day, and nothing from December 2017 reaches training, scaling or model selection.
* Regression or feedforward networks only, no ensembles.
* SHAP explains at least three December predictions, each followed by a written reading.

## 0. Setup

The first cell checks the Python version and installs any missing package, so the notebook can start on
a machine with nothing installed. The second cell defines the constants once, so the test window and the
horizon cannot drift between cells.

In [1]:
import importlib
import importlib.util
import subprocess
import sys

MIN_PYTHON = (3, 9)
REQUIRED = {  # import name -> pip name
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "shap": "shap",
}

if sys.version_info < MIN_PYTHON:
    raise RuntimeError(
        "Python %d.%d or newer is required, found %d.%d. Install a newer Python and rerun."
        % (MIN_PYTHON + tuple(sys.version_info[:2]))
    )

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", *missing],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        # An "externally managed environment" error means the notebook is running in the system Python.
        raise RuntimeError(
            "pip could not install %s. If it reports an externally managed environment, run the notebook "
            "from a virtual environment (python -m venv .venv).\n%s" % (", ".join(missing), result.stderr[-1500:])
        )
    importlib.invalidate_caches()

versions = {name: importlib.import_module(name).__version__ for name in REQUIRED}
print("Python %d.%d.%d | " % tuple(sys.version_info[:3]) + ", ".join("%s %s" % item for item in versions.items()))

Python 3.13.18 | numpy 2.5.5, pandas 3.1.1, sklearn 1.9.1, matplotlib 3.11.2, shap 0.54.1


In [2]:
import os
import urllib.request

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 12)

SEED = 5619
np.random.seed(SEED)

# The three course files are read from disk and never embedded in the notebook.
DATA_DIR = "data"
FILES = {
    "occ": "OccChainletsInTime.txt",
    "amo": "AmoChainletsInTime.txt",
    "price": "pricedBitcoin2009-2018.csv",
}
GITHUB_RAW = "https://raw.githubusercontent.com/cakcora/BitcoinChainnet/master/data/"

# The prediction task, fixed once. December 2017 is the test period; September to November 2017 is the
# validation period used for every modelling choice.
HORIZON_DAYS = 1
TEST_START = pd.Timestamp("2017-12-01")
TEST_END = pd.Timestamp("2017-12-31")
VAL_START = pd.Timestamp("2017-09-01")
VAL_END = pd.Timestamp("2017-11-30")

# The chainlet matrix is 20 x 20; row and column 20 hold the capped cells.
N_CHAINLET = 20


def ensure_data_files(data_dir=DATA_DIR):
    """Return the paths of the three course files, downloading any that are missing."""
    os.makedirs(data_dir, exist_ok=True)
    paths = {}
    for key, name in FILES.items():
        path = os.path.join(data_dir, name)
        if not os.path.exists(path):
            # Download under a temporary name so an interrupted run never leaves a half-written file.
            partial = path + ".part"
            urllib.request.urlretrieve(GITHUB_RAW + name, partial)
            os.replace(partial, path)
        paths[key] = path
        print("%-5s %s  %.1f MB" % (key, path, os.path.getsize(path) / 1e6))
    return paths


PATHS = ensure_data_files()

occ   data/OccChainletsInTime.txt  3.0 MB
amo   data/AmoChainletsInTime.txt  8.0 MB
price data/pricedBitcoin2009-2018.csv  0.1 MB


## 1. Load data and parse dates

The price file's `date` column is parsed once with `pd.to_datetime` and an explicit month/day/year
format. The chainlet files carry only `year` and `day` (the day of the year), so their dates are
assembled from those two numbers with `pd.to_datetime` and a timedelta, with no string handling. Each
loader asserts what it assumes and stops if something is off. The parsed dates are checked against
`year` and `day`, which are then dropped so they cannot reach a model by accident.

In [3]:
def load_price(path):
    """Read the price file and return a frame indexed by date with columns price and totaltx."""
    raw = pd.read_csv(path)
    assert list(raw.columns) == ["date", "price", "year", "day", "totaltx"], "unexpected price columns"

    # An explicit format avoids guessing the day/month order. The year and day checks below confirm it.
    date = pd.to_datetime(raw["date"], format="%m/%d/%Y %H:%M")
    assert date.notna().all()
    assert (date == date.dt.normalize()).all(), "price dates are not all at midnight"
    assert date.is_unique and date.is_monotonic_increasing
    assert (date.dt.year == raw["year"]).all()
    assert (date.dt.day_of_year == raw["day"]).all()
    assert (raw["price"] > 0).all()

    return raw[["price", "totaltx"]].set_index(pd.DatetimeIndex(date, name="date"))


def load_chainlets(path, prefix):
    """Read one chainlet file and return a frame indexed by date.

    Columns are `<prefix>_total` and the 400 cells `<prefix>_<i>_<o>` of the 20 x 20 chainlet matrix.
    """
    raw = pd.read_csv(path, sep="\t")
    cells = ["%d:%d" % (i, o) for i in range(1, N_CHAINLET + 1) for o in range(1, N_CHAINLET + 1)]
    assert list(raw.columns) == ["year", "day", "totaltx"] + cells, "unexpected chainlet columns"
    assert raw[cells].notna().all().all() and (raw[cells] >= 0).all().all()
    # The third column is compared with the cells instead of being trusted by its name.
    assert (raw[cells].sum(axis=1) == raw["totaltx"]).all(), "totaltx is not the sum of the cells"

    # January 1 of `year` plus (day - 1) days. A day number past the end of its year would roll into the
    # next year, which the year check catches.
    january_first = pd.to_datetime(pd.DataFrame({"year": raw["year"], "month": 1, "day": 1}))
    date = january_first + pd.to_timedelta(raw["day"] - 1, unit="D")
    assert date.is_unique and date.is_monotonic_increasing
    assert (date.dt.year == raw["year"]).all()
    assert (date.dt.day_of_year == raw["day"]).all()

    cell_frame = raw[cells].rename(columns=lambda cell: "%s_%s" % (prefix, cell.replace(":", "_")))
    # One concat instead of inserting into a 400-column frame, which pandas flags as fragmented.
    frame = pd.concat([raw["totaltx"].rename("%s_total" % prefix), cell_frame], axis=1)
    frame.index = pd.DatetimeIndex(date, name="date")
    return frame

In [4]:
price = load_price(PATHS["price"])
occ = load_chainlets(PATHS["occ"], "occ")
amo = load_chainlets(PATHS["amo"], "amo")

# December 2017 is read in full: scoring needs the actual prices, and each prediction uses the previous
# day's price.
test_days = pd.date_range(TEST_START, TEST_END, freq="D")
assert len(test_days) == 31 and test_days.isin(price.index).all()

# The days the price file lacks do exist in both chainlet files. Section 2 decides what to do with them.
price_gaps = pd.date_range(price.index.min(), price.index.max(), freq="D").difference(price.index)
assert price_gaps.isin(occ.index).all() and price_gaps.isin(amo.index).all()


def describe_source(name, frame):
    calendar = pd.date_range(frame.index.min(), frame.index.max(), freq="D")
    gaps = calendar.difference(frame.index)
    return {
        "source": name,
        "rows": len(frame),
        "first day": frame.index.min().date(),
        "last day": frame.index.max().date(),
        "index dtype": str(frame.index.dtype),
        "missing days": ", ".join(day.date().isoformat() for day in gaps) or "none",
    }


pd.DataFrame(
    [describe_source("price", price), describe_source("occurrence", occ), describe_source("amount", amo)]
).set_index("source")

,rows,first day,last day,index dtype,missing days
source,,,,,
price,2404,2011-06-01,2017-12-31,datetime64[us],"2012-12-31, 2016-12-31"
occurrence,3281,2009-01-03,2017-12-31,datetime64[us],"2009-01-04, 2009-01-05, 2009-01-06, 2009-01-07"
amount,3281,2009-01-03,2017-12-31,datetime64[us],"2009-01-04, 2009-01-05, 2009-01-06, 2009-01-07"


In [5]:
from IPython.display import display

display(price.head(3))
display(occ.iloc[:3, :4])
display(amo.iloc[:3, :4])

,price,totaltx
date,,
2011-06-01,9.57,6464
2011-06-02,10.60,7632
2011-06-03,14.29,8502


,occ_total,occ_1_1,occ_1_2,occ_1_3
date,,,,
2009-01-03,1,1,0,0
2009-01-08,14,14,0,0
2009-01-09,7,7,0,0


,amo_total,amo_1_1,amo_1_2,amo_1_3
date,,,,
2009-01-03,5000000000,5000000000,0,0
2009-01-08,70000000000,70000000000,0,0
2009-01-09,35000000000,35000000000,0,0


What the checks found, before anything is built on these frames:

* The price file runs from 2011-06-01 to 2017-12-31 (2,404 days), not from 2009 to 2018 as its name
  suggests. It lacks two calendar days, 2012-12-31 and 2016-12-31, both day 366 of a leap year, which is
  what a loop over days 1 to 365 would skip. The chainlet files do have those two days, so the merge in
  Section 2 has to decide what to do with them.
* The chainlet files run from 2009-01-03 to 2017-12-31 (3,281 days) and miss only four days at the very
  start, 2009-01-04 to 2009-01-07. Neither file reaches 2018.
* In both chainlet files the third column, `totaltx`, equals the sum of the 400 cells. In the
  occurrence file it is a transaction count. In the amount file it is a total in satoshis: the first row
  is 5,000,000,000, which is 50 bitcoin and the reward of the genesis block, where the occurrence file
  shows one transaction. The two columns share a name but not a meaning, so they are renamed `occ_total`
  and `amo_total`.
* December 2017 is complete in the price file (31 days). It is read because scoring needs the actual
  prices. It is not used for fitting, scaling or selection.

## 2. Merge occurrence, amount and price

The three frames are joined on a complete daily calendar from the first to the last price date, so every
calendar day is a row and a lag of one row is a lag of one day. Joining on the price dates alone would
skip the two missing days, and a one-row lag across such a gap would span two days.

Those two days keep their chainlet rows and get an empty price. They are not filled forward: a filled
price is an observation nobody made, and the filled day would carry a return of exactly zero. Rows that
need one of these prices, as a lag or as a target, are dropped when the model table is built in Section 3.

Nothing in this section is fitted or estimated, so keeping December 2017 in the frame does not let it
reach training.

In [6]:
def merge_sources(price, occ, amo):
    """Join the three frames on a complete daily calendar over the price period."""
    calendar = pd.date_range(price.index.min(), price.index.max(), freq="D")
    assert occ.index.equals(amo.index), "occurrence and amount files cover different days"
    assert calendar.isin(occ.index).all(), "a chainlet day is missing inside the price period"
    # Reindexing the price onto the calendar keeps its two missing days as empty rows instead of skipping them.
    return pd.concat([price["price"].reindex(calendar), occ.reindex(calendar), amo.reindex(calendar)], axis=1)


data = merge_sources(price, occ, amo)
assert data.index.is_unique and data.index.is_monotonic_increasing

# The price file's transaction count is the occurrence total under another name. Equality on every day shows
# that the three files are aligned on the same calendar date.
count_mismatches = int((price["totaltx"] != data.loc[price.index, "occ_total"]).sum())
assert count_mismatches == 0, "transaction counts disagree, so the dates do not line up"

OCC_CELLS = [name for name in data.columns if name.startswith("occ_") and name != "occ_total"]
AMO_CELLS = [name for name in data.columns if name.startswith("amo_") and name != "amo_total"]
assert len(OCC_CELLS) == len(AMO_CELLS) == N_CHAINLET ** 2

# Only the price may be empty, and only on the days the price file lacks.
assert data.drop(columns="price").notna().all().all()
assert data.index[data["price"].isna()].equals(price_gaps)

# Every score rests on the validation and test months, so they must be complete in every column.
evaluation_days = pd.date_range(VAL_START, TEST_END, freq="D")
assert data.loc[evaluation_days].notna().all().all(), "gaps in the validation or test period"

pd.Series(
    {
        "calendar days": len(data),
        "first day": data.index.min().date(),
        "last day": data.index.max().date(),
        "days with a price": int(data["price"].notna().sum()),
        "days without a price": ", ".join(day.date().isoformat() for day in price_gaps),
        "transaction count mismatches": count_mismatches,
        "occurrence cells": len(OCC_CELLS),
        "amount cells": len(AMO_CELLS),
        "columns": data.shape[1],
    },
    name="value",
).to_frame()

,value
calendar days,2406
first day,2011-06-01
last day,2017-12-31
days with a price,2404
days without a price,"2012-12-31, 2016-12-31"
transaction count mismatches,0
occurrence cells,400
amount cells,400
columns,803


In [7]:
# The gap days keep their chainlet rows and have an empty price.
around_gaps = pd.DatetimeIndex([day + pd.Timedelta(days=offset) for day in price_gaps for offset in (-1, 0, 1)])
data.loc[around_gaps, ["price", "occ_total", "amo_total"]]

,price,occ_total,amo_total
2012-12-30,13.45,43472,86628344663514
2012-12-31,NaN,27844,136824068787140
2013-01-01,13.30,35315,189995377467014
2016-12-30,960.47,288346,207216638799652
2016-12-31,NaN,224275,119964952102904
2017-01-01,997.69,204354,121705100814236


What the merge shows:

* The merged frame has 2,406 rows, one per calendar day from 2011-06-01 to 2017-12-31, and 803 columns:
  the price, the occurrence total with its 400 cells, and the amount total with its 400 cells.
* The price file's transaction count equals the occurrence total on all 2,404 days that have a price. A
  shift of even one day would break that on almost every row, so the price file and the chainlet files
  are aligned on the same calendar date, although their dates were built in two different ways.
* The only empty cells are the prices of 2012-12-31 and 2016-12-31. Both days have chainlet rows, so the
  gaps are in the price file alone.
* September to December 2017 is complete in every column.
* Chainlet history from 2009 to May 2011 is left out. A day with no price can be neither a target nor a
  lag, so that history has no use here.

## 3. Feature engineering

Each row holds what is known at the end of day t, and its target is the move from day t to day t + 1. The
prediction for December 1 therefore sits on the November 30 row, and the December 31 row has no target
because the data end there. That row is never predicted.

The target is the next-day log return, turned back into a price with the actual price on day t. A model
fitted to price levels would have to extrapolate through all of December 2017, when the price sat above
every level in the training years. A return has the same scale in every year, and the rule "tomorrow
equals today" becomes a predicted return of zero.

The features come from the price history, from both chainlet files, and from swing and momentum measures
of the price:

* Price history: seven daily log returns, return volatility over 7 and 30 days, and 30-day momentum.
* Chainlets, for the occurrence file and the amount file separately: the share of the day's total in split
  chainlets (more outputs than inputs), merge chainlets (more inputs than outputs) and the exine cells
  (row or column 20), the shares of the 1:1, 1:2 and 2:2 cells, and the log of the day's total. Shares are
  used instead of raw counts so that the long growth in activity does not dominate. Every summary is taken
  at lags 0, 1 and 2, which are the row's own day and the two days before it.
* Swing and momentum: the position inside the trailing 30 and 90 day high-low range, the distance from the
  0.618 and 0.786 retracement levels of the 90-day range, RSI(14) relative to 50, and whether RSI is
  oversold and how long ago it last was. This is the daily-data version of the levels used in harmonic
  trading.

A few candidates are left out because they are exact combinations of others: the transition share (splits,
merges and transitions add up to one), the mean transaction size (the difference of the two log totals), the
day-over-day change in each log total (a difference of two lags already in the table) and 7-day momentum
(the sum of the seven return lags). A linear model gains nothing from them, and the SHAP values in
Section 7 would divide credit arbitrarily among the features they duplicate.

Price features are built inside each unbroken run of prices, so no window crosses one of the two empty
days from Section 2 and every target is exactly one day after its row. This costs rows, because each run
needs 90 days of history before its longest window is full. The accounting is below.

In [8]:
# Feature settings, defined once so the table and the look-ahead test use the same values.
N_RET_LAGS = 7
VOL_WINDOWS = (7, 30)
MOM_WINDOW = 30
SWING_WINDOWS = (30, 90)
RSI_PERIOD = 14
OVERSOLD_LEVEL = 30
OVERSOLD_CAP = 90
CHAINLET_LAGS = (0, 1, 2)


def rsi(prices, period=RSI_PERIOD):
    """Wilder's RSI. The averages are recursive, so each value depends on past prices only."""
    change = prices.diff()
    gain = change.clip(lower=0)
    loss = -change.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    value = 100 - 100 / (1 + avg_gain / avg_loss)
    # With no losses in the average RSI is 100 by definition, and the ratio above is undefined.
    return value.where(avg_loss > 0, 100.0).where(avg_gain.notna())


def periods_since(flag, cap):
    """Rows since the flag was last true, counted backward only and capped at `cap`."""
    position = np.arange(len(flag))
    last_true = pd.Series(np.where(flag.to_numpy(), position, np.nan), index=flag.index).ffill()
    return (position - last_true).fillna(cap).clip(upper=cap)

In [9]:
def price_block(prices):
    """Price-based features and targets for one unbroken run of consecutive daily prices."""
    log_price = np.log(prices)
    ret = log_price.diff()

    features = pd.DataFrame(index=prices.index)
    for k in range(N_RET_LAGS):
        features["ret_lag%d" % k] = ret.shift(k)
    for window in VOL_WINDOWS:
        features["vol_%d" % window] = ret.rolling(window).std()
    features["mom_%d" % MOM_WINDOW] = log_price - log_price.shift(MOM_WINDOW)

    # Position inside the trailing high-low range: 0 at the low, 1 at the high.
    for window in SWING_WINDOWS:
        high = prices.rolling(window).max()
        low = prices.rolling(window).min()
        features["swing_pos_%d" % window] = (prices - low) / (high - low).replace(0, np.nan)
    # Retracing 61.8% (78.6%) of the range puts the price at position 1 - 0.618 (1 - 0.786).
    features["dist_618_90"] = (features["swing_pos_90"] - (1 - 0.618)).abs()
    features["dist_786_90"] = (features["swing_pos_90"] - (1 - 0.786)).abs()

    strength = rsi(prices)
    known = strength.notna()
    oversold = strength < OVERSOLD_LEVEL
    features["rsi_minus_50"] = strength - 50
    features["rsi_above_50"] = (strength > 50).astype(float).where(known)
    features["oversold"] = oversold.astype(float).where(known)
    features["periods_since_oversold"] = periods_since(oversold, OVERSOLD_CAP).where(known)

    # The targets look one day forward and are never features. The last row of a run has none, because the
    # next price is either empty or past the end of the data.
    targets = pd.DataFrame(
        {
            "price": prices,
            "target_ret": ret.shift(-HORIZON_DAYS),
            "target_price": prices.shift(-HORIZON_DAYS),
        }
    )
    return features, targets


def price_side(price):
    """Run price_block on each unbroken run of prices and stack the results on the full calendar."""
    run_id = price.isna().cumsum()
    blocks = []
    for _, piece in price.groupby(run_id):
        piece = piece.dropna()
        if piece.empty:
            continue
        # Inside a run the rows are consecutive days, so a shift of k rows is a shift of k days.
        assert (piece.index.to_series().diff().dropna() == pd.Timedelta(days=1)).all(), "a run has a missing day"
        blocks.append(price_block(piece))
    features = pd.concat([block[0] for block in blocks]).reindex(price.index)
    targets = pd.concat([block[1] for block in blocks]).reindex(price.index)
    return features, targets

In [10]:
def chainlet_groups(n=N_CHAINLET):
    """Cell names `i_o` of the split, merge and exine groups of the n x n chainlet matrix."""
    cells = [(i, o) for i in range(1, n + 1) for o in range(1, n + 1)]
    return {
        "split": ["%d_%d" % (i, o) for i, o in cells if o > i],
        "merge": ["%d_%d" % (i, o) for i, o in cells if i > o],
        "exine": ["%d_%d" % (i, o) for i, o in cells if i == n or o == n],
    }


def chainlet_features(frame, lags=CHAINLET_LAGS):
    """Daily chainlet summaries of both files, at the given lags in days."""
    groups = chainlet_groups()
    daily = {}
    for prefix in ("occ", "amo"):
        total = frame["%s_total" % prefix]
        for name, cells in groups.items():
            columns = ["%s_%s" % (prefix, cell) for cell in cells]
            daily["%s_share_%s" % (prefix, name)] = frame[columns].sum(axis=1) / total
        for cell in ("1_1", "1_2", "2_2"):
            daily["%s_share_%s" % (prefix, cell)] = frame["%s_%s" % (prefix, cell)] / total
        daily["%s_log_total" % prefix] = np.log(total)
    daily = pd.DataFrame(daily, index=frame.index)
    # The calendar has no missing day, so a shift of k rows is a lag of k days.
    return pd.concat([daily.shift(k).add_suffix("_lag%d" % k) for k in lags], axis=1)


# Split, merge and the diagonal split the 400 cells between them, which is why no transition share is built.
groups = chainlet_groups()
diagonal = ["%d_%d" % (i, i) for i in range(1, N_CHAINLET + 1)]
cell_names = {name.replace("occ_", "", 1) for name in OCC_CELLS}
parts = [set(groups["split"]), set(groups["merge"]), set(diagonal)]
assert len(groups["split"]) == len(groups["merge"]) == N_CHAINLET * (N_CHAINLET - 1) // 2
assert len(groups["exine"]) == 2 * N_CHAINLET - 1
assert sum(len(part) for part in parts) == len(cell_names) and set().union(*parts) == cell_names

In [11]:
def build_table(frame):
    """Features, targets and bookkeeping columns for every calendar day of a merged frame."""
    price_features, targets = price_side(frame["price"])
    table = pd.concat([price_features, chainlet_features(frame), targets], axis=1)
    # The day each target belongs to. It is for splitting and checking only, never a model input.
    target_date = pd.Series(table.index + pd.Timedelta(days=HORIZON_DAYS), index=table.index)
    table["target_date"] = target_date.where(table["target_ret"].notna())
    return table


assert (data[["occ_total", "amo_total"]] > 0).all().all(), "a daily total is zero, so shares are undefined"
table = build_table(data)

BOOKKEEPING = ["price", "target_ret", "target_price", "target_date"]
FEATURES = [name for name in table.columns if name not in BOOKKEEPING]
FAMILIES = {
    "price history": [c for c in FEATURES if c.startswith(("ret_", "vol_", "mom_"))],
    "swing and momentum": [
        c for c in FEATURES if c.startswith(("swing_", "dist_", "rsi_", "oversold", "periods_"))
    ],
    "occurrence chainlets": [c for c in FEATURES if c.startswith("occ_")],
    "amount chainlets": [c for c in FEATURES if c.startswith("amo_")],
}

# Every feature sits in exactly one family, and none is a date, a year, a day or anything of the target.
# The check is on name fragments, so a name such as "days_since" would be caught as well.
assert sum(len(cols) for cols in FAMILIES.values()) == len(FEATURES) == len(set(FEATURES))
assert not [c for c in FEATURES if any(token in c for token in ("year", "day", "date", "target"))]
assert not set(FEATURES) & set(BOOKKEEPING)
# Both chainlet files feed the model.
assert FAMILIES["occurrence chainlets"] and FAMILIES["amount chainlets"]

pd.DataFrame(
    {
        "features": {name: len(cols) for name, cols in FAMILIES.items()},
        "examples": {name: ", ".join(cols[:2]) for name, cols in FAMILIES.items()},
    }
)

,features,examples
price history,10,"ret_lag0, ret_lag1"
swing and momentum,8,"swing_pos_30, swing_pos_90"
occurrence chainlets,21,"occ_share_split_lag0, occ_share_merge_lag0"
amount chainlets,21,"amo_share_split_lag0, amo_share_merge_lag0"


In [12]:
has_target = table["target_ret"].notna()
target_dates = pd.DatetimeIndex(table.loc[has_target, "target_date"])

# The target must be the price file's value on the next calendar day, looked up by date and not by position.
assert (target_dates - table.index[has_target] == pd.Timedelta(days=HORIZON_DAYS)).all()
looked_up = data["price"].reindex(target_dates).to_numpy()
assert np.array_equal(looked_up, table.loc[has_target, "target_price"].to_numpy())
log_move = np.log(table.loc[has_target, "target_price"] / table.loc[has_target, "price"])
assert np.allclose(table.loc[has_target, "target_ret"], log_move)

# The last day of November carries the first day of December as its target, and the last day of the data
# carries none.
alignment_days = pd.DatetimeIndex([VAL_END, TEST_START, TEST_END - pd.Timedelta(days=1), TEST_END])
table.loc[alignment_days, ["price", "target_price", "target_ret", "target_date"]]

,price,target_price,target_ret,target_date
2017-11-30,10147.372,10883.91200,0.070071,2017-12-01
2017-12-01,10883.912,11071.36833,0.017077,2017-12-02
2017-12-30,13215.574,14165.57500,0.069419,2017-12-31
2017-12-31,14165.575,NaN,NaN,NaT


In [13]:
no_price = table["price"].isna()
no_target = ~no_price & table["target_ret"].isna()
warm_up = ~no_price & ~no_target & table[FEATURES].isna().any(axis=1)
complete = ~(no_price | no_target | warm_up)

in_validation = table["target_date"].between(VAL_START, VAL_END)
in_test = table["target_date"].between(TEST_START, TEST_END)
# Every validation day and every December day must have a complete row to be predicted from.
assert in_validation.sum() == complete[in_validation].sum() == 91
assert in_test.sum() == complete[in_test].sum() == 31

# Intermediate results are checked against what they have to look like.
usable = table.loc[complete, FEATURES]
assert np.isfinite(usable.to_numpy()).all()
unit_interval = usable[[c for c in FEATURES if "_share_" in c or c.startswith("swing_pos")]]
assert ((unit_interval >= 0) & (unit_interval <= 1)).all().all()
assert usable["rsi_minus_50"].abs().max() <= 50

# No feature may be an exact combination of the others. Only rows before the validation window are used.
fit_rows = table.loc[complete & (table["target_date"] < VAL_START), FEATURES]
standardized = (fit_rows - fit_rows.mean()) / fit_rows.std()
assert np.linalg.matrix_rank(standardized.to_numpy()) == len(FEATURES)

pd.Series(
    {
        "calendar days": len(table),
        "no price that day": int(no_price.sum()),
        "no price the next day (end of a run)": int(no_target.sum()),
        "windows not yet full (start of a run)": int(warm_up.sum()),
        "complete rows": int(complete.sum()),
        "complete, target before September 2017": int((complete & (table["target_date"] < VAL_START)).sum()),
        "complete, target in the validation window": int(complete[in_validation].sum()),
        "complete, target in December 2017": int(complete[in_test].sum()),
    },
    name="rows",
).to_frame()

,rows
calendar days,2406
no price that day,2
no price the next day (end of a run),3
windows not yet full (start of a run),267
complete rows,2134
"complete, target before September 2017",2012
"complete, target in the validation window",91
"complete, target in December 2017",31


### Proof that no feature looks ahead

Reading the feature code is not enough, so the test breaks the future and checks whether the past notices.
For each cutoff date the features are rebuilt twice. First from data that stop at the cutoff, then from
data in which every value after the cutoff is multiplied by random noise. A centered window, a forward
shift or a statistic taken over the whole sample would change a feature on or before the cutoff, and the
comparison would fail.

In [14]:
def assert_no_lookahead(frame, cutoff, trials=2):
    """Change everything after `cutoff`, rebuild the features and require that nothing up to it moves."""
    clean = frame.astype("float64")
    reference = build_table(clean)
    upto = reference.index <= cutoff
    expected = reference.loc[upto, FEATURES]

    # Truncation: the features up to the cutoff must not need any later day to exist.
    truncated = build_table(clean.loc[clean.index <= cutoff])
    pd.testing.assert_frame_equal(truncated[FEATURES], expected, check_freq=False)

    # Corruption: later values are scaled by random factors, which breaks any hidden use of the future.
    rng = np.random.default_rng(SEED)
    later = ~upto
    for _ in range(trials):
        noisy = clean.copy()
        factors = rng.uniform(0.5, 2.0, size=(int(later.sum()), clean.shape[1]))
        noisy.loc[later, :] = clean.loc[later, :].to_numpy() * factors
        pd.testing.assert_frame_equal(build_table(noisy).loc[upto, FEATURES], expected, check_freq=False)
    return {
        "cutoff": cutoff.date(),
        "rows compared": int(upto.sum()),
        "truncated rebuild": "unchanged",
        "corrupted futures": trials,
        "after corruption": "unchanged",
    }


cutoffs = [pd.Timestamp("2014-06-30"), pd.Timestamp("2017-06-30"), VAL_END, pd.Timestamp("2017-12-15")]
pd.DataFrame([assert_no_lookahead(data, cutoff) for cutoff in cutoffs]).set_index("cutoff")

,rows compared,truncated rebuild,corrupted futures,after corruption
cutoff,,,,
2014-06-30,1126,unchanged,2,unchanged
2017-06-30,2222,unchanged,2,unchanged
2017-11-30,2375,unchanged,2,unchanged
2017-12-15,2390,unchanged,2,unchanged


What the feature table shows:

* There are 60 features: 10 from the price history, 8 swing and momentum measures, and 21 from each
  chainlet file. No feature name carries a date, a year or a day, and none is an exact combination of the
  others, since the standardized features have full rank on the rows before September 2017.
* 2,134 of the 2,406 calendar days have a complete row. The two empty prices cost 182 of the lost rows: the
  two empty days, the two days before them (no next-day price), and the first 89 rows of each run that
  follows. The other 90 are the start of the data and the last day.
* All 91 validation days and all 31 December days have a complete row, and the November 30 row carries the
  December 1 price as its target.
* The look-ahead test passes at four cutoffs, from mid-2014 to the middle of December 2017.
* Neighboring lags of the same chainlet summary are strongly correlated, 0.99 for the log totals. That is
  normal for daily series and not an exact overlap, but a signal in one summary will show up spread over
  its three lags when Section 7 reads the SHAP values.

## 4. Split and scaling

## 5. Model selection

## 6. Final evaluation on December 2017

## 7. Explaining the model with SHAP

## 8. Predictions file

## 9. Compliance audit